In [ ]:
# ライブラリのインポート

import pandas as pd
import torch
from transformers import EsmTokenizer, EsmForProteinFolding
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37
from transformers.models.esm.openfold_utils.protein import Protein as OFProtein, to_pdb

In [ ]:
# デバイスの設定

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# データの読み込み

df = pd.read_csv("../data/processed/amino-acid-genotypes-to-brightness.csv")

In [ ]:
# モデルの読み込み

tokenizer = EsmTokenizer.from_pretrained("facebook/esmfold_v1")
model = (
    EsmForProteinFolding.from_pretrained("facebook/esmfold_v1").eval().to(device)
)

In [ ]:
# 構造の予測

inputs = tokenizer(
    df["sequence"].tolist()[0],
    return_tensors="pt",
    add_special_tokens=False,
    padding=False,
    truncation=False,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

atom_positions = atom14_to_atom37(outputs.positions[-1], outputs)

protein = OFProtein(
    aatype=outputs.aatype[0].cpu().numpy(),
    atom_positions=atom_positions[0].cpu().numpy(),
    atom_mask=outputs.atom37_atom_exists[0].cpu().numpy(),
    residue_index=outputs.residue_index[0].cpu().numpy() + 1,
    b_factors=outputs.plddt[0].cpu().numpy(),
)

pdbstr = to_pdb(protein)

print(pdbstr)